# NIG-TimeNet (PyTorch): one run, data to battery

**Runtime > Run all.** The only cell with settings is the next one. Every option of the run is a dotted key of
`RunSettings` (`nig_timenet_torch/workflow/settings.py`) in `CHANGES`; a misspelt key fails with a suggestion. The cells
below only call the steps of `nig_timenet_torch/workflow/run.py` one by one (the same steps `jobs/baseline.py` calls),
so each result can be looked at before the next step runs.

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "main"
EXPECT_DEVICE = "cuda"                        # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = (("pyarrow", "pyarrow"),)
ARTIFACT = "/content/drive/MyDrive/crypto_model/torch_artifacts/preprocessing_output_1"   # a contract artifact (rule D1)
RUN_DIR = "/content/drive/MyDrive/nig_timenet_runs/notebook-main/train"   # checkpoints: running again resumes
OUT_DIR = "/content/drive/MyDrive/nig_timenet_runs/notebook-main/out"     # reports, battery report, summary
CHANGES = {"train.epochs": 5, "train.batch_size": 256}   # dotted RunSettings keys, e.g. {"split.max_rows": 20000}
BATTERY_FOLDER = ""                            # a finished run folder to read the battery from ("" = skip section 5)
BATTERY_COST = 0.0008                          # cost PER SIDE; a trade pays it twice
BATTERY_VARIANTS = 1                           # variants tried so far (widens the confidence intervals)

## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Settings
The artifact and the run folder go into `RunSettings`, `CHANGES` is applied on top (unknown keys raise).

In [ ]:
from nig_timenet_torch.core.config import updated
from nig_timenet_torch.data_io.loading import LoaderSettings
from nig_timenet_torch.workflow.settings import RunSettings
settings = updated(RunSettings(run_dir=RUN_DIR, data=LoaderSettings(name=ARTIFACT)), CHANGES)
print(settings)

## 3. Data and split
Loads the artifact (refused unless it is a contract artifact), reads its facts and splits by time with purge.

In [ ]:
from nig_timenet_torch.workflow import run as steps
run = steps.prepare(settings, OUT_DIR)
print({k: len(v) for k, v in run.idx.items()}, "train_end / val_end:", run.bounds)

## 4. Model and training
`build` seeds and builds the model, the trainer and the run identity; `train` resumes a run folder that has a checkpoint.

In [ ]:
steps.build(run)
print(run.ms.architecture, "| parameters:", sum(p.numel() for p in run.trainer.model.parameters()))

In [ ]:
from nig_timenet_torch.workflow.flat_run import epoch_line
steps.train(run, on_epoch=lambda row: print(epoch_line(row, settings.train.epochs)))

## 5. Reports
Health report (validation), test report (test split, best-epoch weights) and the evaluation battery on val and test
(cost per side; direction is the close class head).

In [ ]:
steps.health_step(run)
print(open(OUT_DIR + "/health_report.md", encoding="utf-8").read()[:3000])

In [ ]:
report = steps.test_step(run)
print("\n".join(steps.test_lines(report)))

In [ ]:
from nig_timenet_torch.workflow.battery_run import render
sections = steps.battery_step(run)
print(render(sections))

In [ ]:
summary = steps.write_summary(run)
print(summary["epochs_done"], "epochs;", "outputs in", OUT_DIR)

## 6. Battery of a finished run folder (any model)
Reads `battery_table_{val,test}.parquet` of a folder a run wrote and runs the whole battery again, with another cost or
more variants if you like. No model and no artifact needed. Set `BATTERY_FOLDER` in the settings cell.

In [ ]:
from nig_timenet_torch.workflow.battery_run import battery_from_folder, render
if BATTERY_FOLDER:
    print(render(battery_from_folder(BATTERY_FOLDER, cost=BATTERY_COST, n_variants=BATTERY_VARIANTS)))